In [1]:
# [CELL 1] STUDENT CONFIGURATION, DATA LOADING, AND FEATURE SCREENING
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Pacudan"
STUDENT_ID = "TUPM-26-1859"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

# Find the SECOM data folder in this folder or any parent folder
WORK_DIR = Path.cwd()
OUTPUT_DIR = WORK_DIR


def find_data_dir():
    for base in [WORK_DIR, *WORK_DIR.parents]:
        if (base / "data" / "secom.data").is_file():
            return base / "data"
    raise FileNotFoundError("data/secom.data not found in this folder or any parent folder.")

DATA_DIR = find_data_dir()
measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw.ndim == 2 and len(labels) == len(raw)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])
is_pass = labels == -1
is_fail = labels == 1

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)
print("Workspace:", WORK_DIR.resolve())
print("Data Folder:", DATA_DIR.resolve())
print("Output Folder:", OUTPUT_DIR.resolve())
print("Measurement Matrix Shape:", raw.shape)
print("Pass Records:", int(is_pass.sum()), "| Fail Records:", int(is_fail.sum()))
# Feature screening shared by the additional exercises
missing_mask = np.isnan(raw)
observed_n = (~missing_mask).sum(axis=0)
obs_min = np.where(missing_mask, np.inf, raw).min(axis=0)
obs_max = np.where(missing_mask, -np.inf, raw).max(axis=0)

has_missing = missing_mask.any(axis=0)
all_missing = observed_n == 0
constant = (observed_n > 0) & (obs_min == obs_max)
usable = ~(all_missing | constant)
usable_idx = np.flatnonzero(usable)
print("Usable Features:", len(usable_idx))


Student: Pacudan TUPM-26-1859
Seed Digit: 9
ID Sum: 31
Name Length: 7
Random Seed: 261859
Verification Value: 226
Workspace: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\Lab7_NumPy_Pandas_Additional_Exercises
Data Folder: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\data
Output Folder: C:\Users\jullm\Lab7_NumPy_Pandas_Pacudan\Lab7_NumPy_Pandas_Additional_Exercises
Measurement Matrix Shape: (1567, 590)
Pass Records: 1463 | Fail Records: 104
Usable Features: 474


In [2]:
# [CELL 2] EXERCISE 3 - ROBUSTNESS AND SENSITIVITY ANALYSIS
# Requirement 1: five usable features with at least 10 observed values in each outcome group
obs_pass = (~np.isnan(raw[is_pass])).sum(axis=0)
obs_fail = (~np.isnan(raw[is_fail])).sum(axis=0)
eligible = np.flatnonzero(usable & (obs_pass >= 10) & (obs_fail >= 10))

rng3 = np.random.default_rng(RANDOM_SEED)
pos3 = np.sort(rng3.choice(eligible, size=5, replace=False))
names3 = feature_names[pos3].tolist()

P = raw[is_pass][:, pos3]
F = raw[is_fail][:, pos3]

# Requirement 2: two preparation strategies
# A = available-case analysis, B = median imputation using full-dataset medians
full_med3 = np.nanmedian(raw[:, pos3], axis=0)
P_imp = np.where(np.isnan(P), full_med3, P)
F_imp = np.where(np.isnan(F), full_med3, F)

# Requirement 3: Fail minus Pass mean difference for each strategy
diff_A = np.nanmean(F, axis=0) - np.nanmean(P, axis=0)
diff_B = F_imp.mean(axis=0) - P_imp.mean(axis=0)

# Requirement 4: stratified bootstrap, 200 + 10 x SEED_DIGIT repetitions
N_BOOT = 200 + 10 * SEED_DIGIT

def bootstrap(seed):
    rng = np.random.default_rng(seed)
    boot_A = np.empty((N_BOOT, len(pos3)))
    boot_B = np.empty((N_BOOT, len(pos3)))
    for b in range(N_BOOT):
        ip = rng.integers(0, len(P), len(P))   # resample Pass records with replacement
        iF = rng.integers(0, len(F), len(F))   # resample Fail records independently
        boot_A[b] = np.nanmean(F[iF], axis=0) - np.nanmean(P[ip], axis=0)
        boot_B[b] = F_imp[iF].mean(axis=0) - P_imp[ip].mean(axis=0)
    return boot_A, boot_B

boot_A, boot_B = bootstrap(RANDOM_SEED)

# Requirement 5: bootstrap mean difference and percentile 95% confidence interval
def summarize(strategy, observed, boot, n_pass, n_fail):
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    out = pd.DataFrame({
        "Feature": names3,
        "Strategy": strategy,
        "Pass_N": n_pass,
        "Fail_N": n_fail,
        "Observed_Mean_Diff": observed,
        "Bootstrap_Mean_Diff": boot.mean(axis=0),
        "CI_Lower_95": lo,
        "CI_Upper_95": hi,
    })
    out["Direction"] = np.where(out["Observed_Mean_Diff"] > 0, "Fail higher",
                         np.where(out["Observed_Mean_Diff"] < 0, "Pass higher", "No difference"))
    out["CI_Includes_Zero"] = (out["CI_Lower_95"] <= 0) & (out["CI_Upper_95"] >= 0)
    out["Rank_By_Abs_Diff"] = out["Observed_Mean_Diff"].abs().rank(ascending=False, method="min").astype(int)
    return out

res_A = summarize("Available-case", diff_A, boot_A, (~np.isnan(P)).sum(axis=0), (~np.isnan(F)).sum(axis=0))
res_B = summarize("Median-imputed", diff_B, boot_B, len(P), len(F))
robust = pd.concat([res_A, res_B], ignore_index=True)

# Requirement 6: compare direction, confidence intervals, and rankings across strategies
sign_changed = np.sign(res_A["Observed_Mean_Diff"].to_numpy()) != np.sign(res_B["Observed_Mean_Diff"].to_numpy())
rank_changed = res_A["Rank_By_Abs_Diff"].to_numpy() != res_B["Rank_By_Abs_Diff"].to_numpy()
ci_changed = res_A["CI_Includes_Zero"].to_numpy() != res_B["CI_Includes_Zero"].to_numpy()
change = np.abs(diff_B - diff_A)
k = int(np.argmax(change))

# Requirement 7: reproducibility with the same seed, then export
boot_A2, boot_B2 = bootstrap(RANDOM_SEED)
repro_ok = bool(np.array_equal(boot_A, boot_A2) and np.array_equal(boot_B, boot_B2))

robust_path = OUTPUT_DIR / "robustness_comparison.csv"
report_path = OUTPUT_DIR / "sensitivity_report.txt"
robust.to_csv(robust_path, index=False)

def names_or_none(mask):
    return ", ".join(np.array(names3)[mask]) if mask.any() else "None"

report = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Selected Features: {', '.join(names3)}\n"
    f"Bootstrap Repetitions: {N_BOOT}\n"
    f"Largest Change in Mean Difference: {names3[k]} ({change[k]:.4f})\n"
    f"Features with Changed Difference Sign: {names_or_none(sign_changed)}\n"
    f"Features with Changed Ranking: {names_or_none(rank_changed)}\n"
    f"Features with Changed CI Zero-Inclusion: {names_or_none(ci_changed)}\n"
    f"Reproducibility Verified: {repro_ok}\n\n"
    "Interpretation Notes:\n"
    "- Available-case analysis uses only observed values, while median imputation fills gaps with the full-dataset median.\n"
    "- A confidence interval that includes zero means the Fail-Pass difference is not clearly separated from zero.\n"
    "- Rankings use raw mean differences, so features with larger measurement scales can rank higher.\n"
    "- Consistent results across both strategies make a finding more reliable, but they do not show failure cause.\n"
)
report_path.write_text(report, encoding="utf-8")

print("Selected Features:", names3)
print("Bootstrap Repetitions:", N_BOOT)
print()
print(robust.round(4).to_string(index=False))
print()
print("=" * 70)
print("EXERCISE 3 - ASSESSMENT DATA")
print("=" * 70)
print(f"{'Selected Features':<34}: {', '.join(names3)}")
print(f"{'Bootstrap Repetitions':<34}: {N_BOOT}")
print(f"{'Largest Change in Mean Difference':<34}: {names3[k]} ({change[k]:.4f})")
print(f"{'Features with Changed Sign':<34}: {int(sign_changed.sum())} ({names_or_none(sign_changed)})")
print(f"{'Features with Changed Ranking':<34}: {int(rank_changed.sum())} ({names_or_none(rank_changed)})")
print(f"{'Reproducibility Verified':<34}: {repro_ok}")
print(f"{'Required Outputs':<34}: {robust_path.name}, {report_path.name} "
      f"(created: {robust_path.is_file() and report_path.is_file()})")
print("=" * 70)

Selected Features: ['Feature_152', 'Feature_386', 'Feature_467', 'Feature_472', 'Feature_568']
Bootstrap Repetitions: 290

    Feature       Strategy  Pass_N  Fail_N  Observed_Mean_Diff  Bootstrap_Mean_Diff  CI_Lower_95  CI_Upper_95   Direction  CI_Includes_Zero  Rank_By_Abs_Diff
Feature_152 Available-case    1460     104             -0.4322              -0.5064      -1.7940       0.3455 Pass higher              True                 2
Feature_386 Available-case    1463     104              0.0005               0.0005      -0.0012       0.0031 Fail higher              True                 5
Feature_467 Available-case    1460     103              1.2771               1.1481      -0.9882       4.2078 Fail higher              True                 1
Feature_472 Available-case    1456     104              0.2331               0.4867      -8.8060       9.1755 Fail higher              True                 3
Feature_568 Available-case    1214      80              0.0782               0.0888    